# 18 · Checkpoint: tune an engine to a TTFT target
Level L3 · Part 2 · core · ~25 min · builds on 14, 16

Time: about 25 min to read. The live run starts the pinned vLLM CPU image twice. On this 4-core lab machine the first request of each new server takes about 10 minutes (a one-time start-up cost), so a live run takes about 25 minutes. Without Docker, the cells print the recorded run of the same image, labelled as such.

## Goal
By the end of this checkpoint you can tune one vLLM server to a time-to-first-token (TTFT) target on the Brightside policy prompt, measure two configurations with the same streamed requests, and justify each flag with a number the engine reports.

## Why it matters
- **Without a measured target:** you change flags by guess. On this CPU, a prompt of about 1,450 tokens (the policy and a question) takes about 21 seconds to its first token with no prefix cache (step 3 measures it). A flag that looks right on paper can still miss the target.
- **With it:** one target number, two measured configurations, and one decision you can defend to a reviewer.
- **When you don't need it:** no latency target at all, or a one-user demo. Keep the defaults.

## The idea
TTFT is the wait from sending a request to its first token. It is the queue wait plus the prefill: the engine must compute the KV cache for every prompt token it has not seen before. Every Brightside request starts with the same policy, about 1,400 tokens. The prefix cache (lesson 14) keeps the KV blocks of that policy, so after the first request a new request computes only its own question. The lesson 16 flags keep the cache sized and the batch bounded. Like a kitchen that keeps the stock for a standing order ready: the first order is slow, and the rest only cook the new part.

```mermaid
flowchart LR
  T["Target: TTFT p50 at most 2.0 s"] --> A["Config A: prefix caching off"]
  T --> B["Config B: lesson 16 flags, prefix caching on"]
  A --> MA["warm-up, then 4 timed requests"]
  B --> MB["warm-up, then 4 timed requests"]
  MA --> D{"p50 at most 2.0 s?"}
  MB --> D
  D --> R["Report: the flags, their numbers, the decision"]
```

## Requirements
- [ ] The target is one number: TTFT p50 at most 2.0 s for the policy prompt plus a question, on this lab machine.
- [ ] Two configurations are measured with the same questions: prefix caching off (A) and on (B).
- [ ] Each configuration reads its effective values back from the engine: the prefix caching switch, the KV cache size and the prefix-cache hit rate.
- [ ] The configuration you choose meets the target, and each flag has a number beside it.
- [ ] The decision is one sentence you could defend in a review.

## Build it
### Step 1: the target and the prompts
The policy text is made up: the course has no copy of a real Brightside policy. A fixed seed makes the same 50 rules in every run. The trace from lesson 07 gives the longest request, which lesson 16 already fitted into the length flag. Its prompts run to about 1,600 tokens; this checkpoint uses the policy plus a short question, about 1,450 tokens.

In [1]:
import json, random, re, sys
from pathlib import Path

TTFT_TARGET_S = 2.0                                       # the target: p50 time to first token, lab CPU
trace = [json.loads(line) for line in Path("../data/07/support_trace.jsonl").read_text().splitlines()]
longest = max(r["prompt_tokens"] for r in trace)
rng = random.Random(18)                                   # fixed seed: the same policy text in every run
TOPICS = ["refund", "delivery", "warranty", "password reset", "billing date", "gift card", "exchange"]
POLICY = "\n".join(f"Rule {i}: for a {rng.choice(TOPICS)} request, check the order date, confirm the account "
                   f"owner, and answer within {rng.randint(1, 10)} business days." for i in range(1, 51))
QUESTIONS = ["Can I return a gift card?", "Where is my parcel?", "My password reset email is late.",
             "Can I change my billing date?", "The exchange item arrived damaged.",
             "How long does a warranty claim take?", "Can I get a refund without the receipt?",
             "Why was my account locked?"]
print(f"target: TTFT p50 at most {TTFT_TARGET_S} s; longest trace request: {longest:,} tokens")
print(f"made-up policy: {len(POLICY.split()):,} words; step 3 reports the prompt tokens the engine counts")

target: TTFT p50 at most 2.0 s; longest trace request: 1,617 tokens
made-up policy: 1,023 words; step 3 reports the prompt tokens the engine counts


### Step 2: the helpers
These are the start and read helpers of lesson 16, shortened. The server takes the lesson 16 target flags and the extra flags of one configuration. The key is optional: unset, the server takes calls without one, and the value is never printed.

In [2]:
import os, socket, subprocess, time
import httpx
from openai import OpenAI
sys.path.insert(0, "..")                                  # the course folder, so tut_env.py can be imported
from tut_env import env

NAME, PORT = "tut-kv-forge-15h-18", 8180                  # this lesson's container and port
BASE = f"http://127.0.0.1:{PORT}"
IMAGE, SERVED = "vllm/vllm-openai-cpu:v0.32.0", "brightside-qwen-0.5b"    # pinned, as in lessons 08 to 16
KEY = env("VLLM_API_KEY", "")
TARGET = ["Qwen/Qwen2.5-0.5B-Instruct", "--served-model-name", SERVED, "--host", "0.0.0.0",
          "--dtype", "float32", "--enforce-eager", "--max-model-len", "2048", "--max-num-seqs", "4",
          "--max-num-batched-tokens", "2048", "--kv-cache-memory-bytes", str(2 * 1024**3)]

def server_up():
    try:
        return httpx.get(f"{BASE}/health", timeout=2).status_code == 200
    except httpx.TransportError:
        return False

The helpers below start the container, wait for its health route, and stop it. They are the lesson 16 helpers.

In [3]:
def start(extra):
    """Start the container with the lesson 16 flags plus `extra`; wait for the health route."""
    subprocess.run(["docker", "rm", "-f", NAME], capture_output=True)
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", PORT)) == 0:
            raise RuntimeError(f"port {PORT} is busy: stop the other process first")
    run_env, key_flag = dict(os.environ), []
    if KEY:
        run_env["VLLM_API_KEY"], key_flag = KEY, ["-e", "VLLM_API_KEY"]
    cmd = ["docker", "run", "-d", "--name", NAME, "-p", f"{PORT}:8000", "--shm-size", "4g",
           "-v", f"{Path.home() / '.cache' / 'huggingface'}:/root/.cache/huggingface", *key_flag, IMAGE, *TARGET, *extra]
    if subprocess.run(cmd, capture_output=True, text=True, env=run_env, timeout=120).returncode != 0:
        raise RuntimeError("docker run failed")
    deadline = time.time() + 480
    while time.time() < deadline and not server_up():
        time.sleep(2)
    if not server_up():
        raise TimeoutError(f"server not healthy after 480 s on port {PORT}")

def stop():
    subprocess.run(["docker", "rm", "-f", NAME], capture_output=True)

print("helpers ready; server answering now:", server_up())

helpers ready; server answering now: False


### Step 3: configuration A, prefix caching off
Start the server with `--no-enable-prefix-caching`. The start takes a minute or two: the engine loads the weights and sizes the cache. The next cells then do the work: one warm-up request, then three timed requests, each with the same policy and a new question. The warm-up is not timed: on this CPU it takes about 10 minutes, the one-time cost of a new server. The timed requests take about 20 seconds each.

In [4]:
start(["--no-enable-prefix-caching"])
print("configuration A: server answering:", server_up())

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


TTFT is the time from sending the request to the first token that has text. Prefix-cache counters are read before and after the timed requests, so the hit rate covers only those requests. The cell stops the container at the end.

In [5]:
def ttft(client, question):
    """Seconds from sending a streamed request to its first token with text."""
    t0, first = time.perf_counter(), None
    stream = client.chat.completions.create(model=SERVED, max_tokens=2, temperature=0, stream=True, timeout=900,
                                            messages=[{"role": "user", "content": POLICY + "\n\nQuestion: " + question}])
    for chunk in stream:
        if first is None and chunk.choices and chunk.choices[0].delta.content:
            first = time.perf_counter() - t0
    return first

def counts():
    """Prefix-cache hits and queries so far, and the /metrics text."""
    text = httpx.get(f"{BASE}/metrics", timeout=30).text
    total = lambda name: sum(float(l.split()[-1]) for l in text.splitlines() if l.startswith((name + "{", name + " ")))
    return total("vllm:prefix_cache_hits_total"), total("vllm:prefix_cache_queries_total"), text

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


`measure` runs the warm-up and the timed requests, then reads the engine's own account and stops the server.

In [6]:
def measure(label):
    """Warm-up, three timed requests, then the engine's own account. Stops the server at the end."""
    try:
        client = OpenAI(base_url=f"{BASE}/v1", api_key=KEY or "not-needed")
        warm = client.chat.completions.create(model=SERVED, max_tokens=1, temperature=0, timeout=900,
                                              messages=[{"role": "user", "content": POLICY + "\n\nQuestion: " + QUESTIONS[0]}])
        h0, q0, _ = counts()
        times = [ttft(client, q) for q in QUESTIONS[1:4]]
        h1, q1, text = counts()
        logs = subprocess.run(["docker", "logs", NAME], capture_output=True, text=True)
        kv = re.search(r"KV cache size: ([\d,]+) tokens", logs.stdout + logs.stderr).group(1)
        return {"label": label, "ttft_s": [round(t, 3) for t in times],
                "p50": round(sorted(times)[1], 3), "max": round(max(times), 3),
                "hit_rate": round((h1 - h0) / (q1 - q0), 3) if q1 > q0 else 0.0,
                "prefix_caching": re.search(r'enable_prefix_caching="(\w+)"', text).group(1),
                "kv_tokens": int(kv.replace(",", "")), "prompt_tokens": warm.usage.prompt_tokens}
    finally:
        stop()

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Run configuration A. This cell takes about 12 minutes on this CPU.

In [7]:
A = measure("A: prefix caching off")
print(A)

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


### Step 4: configuration B, the lesson 16 flags with prefix caching on
Same flags, same questions, same warm-up. Only the prefix switch changes: prefix caching is on by default, so the extra flags are empty. Start the server first.

In [8]:
start([])
print("configuration B: server answering:", server_up())

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


The same measurement as step 3. The warm-up now also fills the prefix cache with the policy blocks, so the timed requests can reuse them.

In [9]:
B = measure("B: lesson 16 flags")
print(B)

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**If Docker was not run**, the next cell loads the recorded run of the same image (`data/recorded/18.json`) and says so.

In [10]:
Path("../data/18").mkdir(parents=True, exist_ok=True)
if "B" not in globals():
    rec = json.loads(Path("../data/recorded/18.json").read_text())
    A, B = rec["A"], rec["B"]
    print("source: recorded output,", rec["note"])
else:
    print("source: this run")
    Path("../data/18/results.json").write_text(json.dumps({"A": A, "B": B}, indent=1))
for r in (A, B):
    print(f"{r['label']:26} p50 {r['p50']:6.2f} s  max {r['max']:6.2f} s  hit rate {r['hit_rate']:.2f}  "
          f"prefix caching {r['prefix_caching']}  KV tokens {r['kv_tokens']:,}  meets target: {r['p50'] <= TTFT_TARGET_S}")

source: recorded output, a live run of this notebook on the course machine (Docker, pinned CPU image, 4 CPU cores, no GPU), 2026-10-22; the first request of each server took about 10 minutes
A: prefix caching off      p50  21.17 s  max  22.37 s  hit rate 0.00  prefix caching False  KV tokens 87,296  meets target: False
B: lesson 16 flags         p50   1.13 s  max   1.37 s  hit rate 0.97  prefix caching True  KV tokens 87,296  meets target: True


### Step 5: your decision
**Your task:** write `chosen`, the label of the configuration you keep. It must be the one that meets the target, and if both meet it, the one with the lower p50. Then print one line that names the flag that made the difference and its measured number. If neither meets the target, set `chosen` to `None` and print what you would change next.

In [11]:
# your code here

## Reference solution
**Try step 5 first.** The answer is in the hidden cell below.

In [12]:
ok = [r for r in (A, B) if r["p50"] <= TTFT_TARGET_S]
chosen = min(ok, key=lambda r: r["p50"]) if ok else None
assert A["prefix_caching"] == "False" and B["prefix_caching"] == "True", "the engine reports the wrong switch"
print("chosen:", chosen["label"] if chosen else None)
print(f"prompt: {B['prompt_tokens']:,} tokens, of which the policy is the shared prefix")
print(f"the prefix cache moved p50 from {A['p50']:.2f} s to {B['p50']:.2f} s; hit rate {B['hit_rate']:.2f}")
print(f"memory and length flags: {B['kv_tokens']:,} KV tokens, the same as lesson 16 (87,296)")
# Recorded run: A p50 21.17 s (misses the 2.0 s target); B p50 1.13 s (meets it), hit rate 0.97, 87,296 KV tokens.
# The decision: keep prefix caching on and keep the lesson 16 flags.

chosen: B: lesson 16 flags
prompt: 1,454 tokens, of which the policy is the shared prefix
the prefix cache moved p50 from 21.17 s to 1.13 s; hit rate 0.97
memory and length flags: 87,296 KV tokens, the same as lesson 16 (87,296)


## Key terms
- **TTFT** — time to first token: the wait from sending a request to its first token.
- **Prefill** — the engine's work to compute KV for the prompt before it can produce a token.
- **Prefix cache hit rate** — the share of prompt blocks found in the cache (lesson 14).
- **p50** — the value that half of the timed requests stay under.

**Interview:** "A chat assistant with a long shared system prompt has a TTFT of 20 seconds. What do you change, and how do you show it worked?" Strong answer: turn on prefix caching, check the hit rate and the KV budget in the engine's own counters, then measure p50 TTFT on the same requests before and after.